# Duke Dining Chatbot — Data Prep
**Goal:** Clean the raw CSV data, merge menus with location info, generate sentence embeddings, and save everything to `embeddings.pkl` for use in the chatbot.

Steps:
1. Install dependencies
2. Load & inspect data
3. Clean & preprocess
4. Build rich text blobs for each menu item
5. Generate embeddings with `sentence-transformers`
6. Save to `embeddings.pkl`
7. Sanity check — test a query

## Step 1 — Install Dependencies

In [1]:
# Run once — Colab already has pandas/numpy, just need sentence-transformers
!pip install sentence-transformers --quiet

## Step 2 — Load & Inspect Data

In [2]:
import pandas as pd
import numpy as np

# -----------------------------------------------------------
# Upload your two CSVs to Colab using the file sidebar,
# or mount Google Drive and adjust the paths below.
# -----------------------------------------------------------

# Option A: Upload directly to Colab session
# from google.colab import files
# files.upload()   # select both CSVs

# Option B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# MENU_PATH = '/content/drive/MyDrive/duke-dining-chatbot/data/menu_items_tagged.csv'
# LOC_PATH  = '/content/drive/MyDrive/duke-dining-chatbot/data/WU_locations_Updated.csv'

# Option C: Paths after direct upload
MENU_PATH = 'menu_items_retagged.csv'
LOC_PATH  = 'WU_locations_Updated.csv'

menu_df = pd.read_csv(MENU_PATH)
loc_df  = pd.read_csv(LOC_PATH)

print('=== Menu Items ===')
print(f'Shape: {menu_df.shape}')
print(menu_df.head(3).to_string())
print()
print('=== Locations ===')
print(f'Shape: {loc_df.shape}')
print(loc_df.head(3).to_string())

=== Menu Items ===
Shape: (394, 6)
   item_id  location_id                               name meal_period                                       description                 generated_tags
0        1            1    Bacon, Egg, and Cheese Sandwich   Breakfast    Breakfast sandwich with bacon, egg, and cheese   breakfast,sandwich,dairy,egg
1        2            1            Egg and Cheese Sandwich   Breakfast            Breakfast sandwich with egg and cheese  breakfast,sandwich,eggs,dairy
2        3            1  Sausage, Egg, and Cheese Sandwich   Breakfast  Breakfast sandwich with sausage, egg, and cheese  breakfast,sandwich,eggs,dairy

=== Locations ===
Shape: (16, 8)
   location_id              name       campus      location                                                    hours   cuisine                                                                                                                                                                                       description   

In [3]:
# Quick data quality check
print('--- Null counts: menu ---')
print(menu_df.isnull().sum())
print()
print('--- Null counts: locations ---')
print(loc_df.isnull().sum())
print()
print('--- Unique meal periods ---')
print(menu_df['meal_period'].value_counts())
print()
print('--- Unique location names ---')
print(loc_df['name'].tolist())

--- Null counts: menu ---
item_id           0
location_id       0
name              0
meal_period       0
description       0
generated_tags    0
dtype: int64

--- Null counts: locations ---
location_id    0
name           0
campus         0
location       0
hours          0
cuisine        0
description    0
tags           0
dtype: int64

--- Unique meal periods ---
meal_period
lunch, dinner                239
Drink                         27
drink                         24
lunch, dinner, snack          23
breakfast                     19
dinner                        17
dessert                       12
snack, dessert                 9
breakfast, dessert, lunch      6
breakfast, snack, dessert      5
Breakfast                      3
breakfast, lunch               3
snack                          2
breakfast, snack               2
lunch, snack                   1
breakfast, snack               1
lunch                          1
Name: count, dtype: int64

--- Unique location names ---
[

## Step 3 — Clean & Preprocess
*(This is the preprocessing pipeline that satisfies the rubric item for data quality handling.)*

In [4]:
import re

def clean_text(text: str) -> str:
    """Lowercase, strip extra whitespace, remove special characters."""
    if not isinstance(text, str):
        return ''
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)          # collapse whitespace
    text = re.sub(r'[^\w\s,./\'-]', '', text) # keep alphanumeric + basic punctuation
    return text

def normalize_tags(tags: str) -> list:
    """Split comma-separated tags into a clean list."""
    if not isinstance(tags, str):
        return []
    return [t.strip().lower() for t in tags.split(',') if t.strip()]

# --- Clean menu data ---
menu_df['name_clean']        = menu_df['name'].apply(clean_text)
menu_df['description_clean'] = menu_df['description'].apply(clean_text)
menu_df['meal_period_clean'] = menu_df['meal_period'].apply(clean_text)
menu_df['tags_list']         = menu_df['generated_tags'].apply(normalize_tags)

# --- Clean location data ---
loc_df['name_clean']        = loc_df['name'].apply(clean_text)
loc_df['description_clean'] = loc_df['description'].apply(clean_text)
loc_df['cuisine_clean']     = loc_df['cuisine'].apply(clean_text)
loc_df['tags_list']         = loc_df['tags'].apply(normalize_tags)

print('Cleaning done.')
print(menu_df[['name_clean', 'description_clean', 'tags_list']].head(3).to_string())

Cleaning done.
                          name_clean                                 description_clean                           tags_list
0    bacon, egg, and cheese sandwich    breakfast sandwich with bacon, egg, and cheese   [breakfast, sandwich, dairy, egg]
1            egg and cheese sandwich            breakfast sandwich with egg and cheese  [breakfast, sandwich, eggs, dairy]
2  sausage, egg, and cheese sandwich  breakfast sandwich with sausage, egg, and cheese  [breakfast, sandwich, eggs, dairy]


## Step 4 — Merge & Build Text Blobs
Each menu item gets enriched with its location info, then converted into a single descriptive string that the embedding model will encode. Rich text = better semantic search.

In [5]:
# Merge menu items with their location info
merged_df = menu_df.merge(
    loc_df[['location_id', 'name', 'name_clean', 'cuisine_clean',
            'description_clean', 'tags_list', 'hours', 'location']],
    on='location_id',
    suffixes=('_item', '_location')
)

print(f'Merged shape: {merged_df.shape}')
print(merged_df.columns.tolist())

Merged shape: (394, 17)
['item_id', 'location_id', 'name_item', 'meal_period', 'description', 'generated_tags', 'name_clean_item', 'description_clean_item', 'meal_period_clean', 'tags_list_item', 'name_location', 'name_clean_location', 'cuisine_clean', 'description_clean_location', 'tags_list_location', 'hours', 'location']


In [7]:
def build_text_blob(row) -> str:
    """
    Build a rich descriptive string for each menu item.
    This is what gets embedded — more detail = better retrieval.
    Format: natural sentence so the embedding model handles it well.
    """
    item_tags = ' '.join(row['tags_list_item'])           # e.g. 'breakfast dairy comfort_food'
    loc_tags  = ' '.join(row['tags_list_location'])       # e.g. 'coffee drinks grab-and-go'

    blob = (
        f"{row['name_clean_item']} is a {row['meal_period_clean']} item "
        f"at {row['name_clean_location']}, a {row['cuisine_clean']} dining spot "
        f"located in {row['location']}. "
        f"Description: {row['description_clean_item']}. "
        f"Tags: {item_tags}. "
        f"Venue tags: {loc_tags}. "
        f"Hours: {row['hours']}."
    )
    return blob

merged_df['text_blob'] = merged_df.apply(build_text_blob, axis=1)

# Preview a few blobs
for blob in merged_df['text_blob'].head(3):
    print(blob)
    print()

bacon, egg, and cheese sandwich is a breakfast item at beyu blue coffee, a cafe dining spot located in Bryan Center. Description: breakfast sandwich with bacon, egg, and cheese. Tags: breakfast sandwich dairy egg. Venue tags: coffee drinks breakfast grab-and-go quick. Hours: 7 am - 7 pm (mon - fri); 9 am - 5 pm (sat - sun).

egg and cheese sandwich is a breakfast item at beyu blue coffee, a cafe dining spot located in Bryan Center. Description: breakfast sandwich with egg and cheese. Tags: breakfast sandwich eggs dairy. Venue tags: coffee drinks breakfast grab-and-go quick. Hours: 7 am - 7 pm (mon - fri); 9 am - 5 pm (sat - sun).

sausage, egg, and cheese sandwich is a breakfast item at beyu blue coffee, a cafe dining spot located in Bryan Center. Description: breakfast sandwich with sausage, egg, and cheese. Tags: breakfast sandwich eggs dairy. Venue tags: coffee drinks breakfast grab-and-go quick. Hours: 7 am - 7 pm (mon - fri); 9 am - 5 pm (sat - sun).



## Step 5 — Generate Embeddings
`all-MiniLM-L6-v2` is a lightweight model (80MB) that runs fast on CPU and is specifically trained for semantic similarity tasks. It converts each text blob into a 384-dimensional vector.

In [8]:
from sentence_transformers import SentenceTransformer

print('Loading embedding model...')
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded.')

text_blobs = merged_df['text_blob'].tolist()

print(f'Generating embeddings for {len(text_blobs)} menu items...')
# show_progress_bar=True so you can see it working
embeddings = model.encode(text_blobs, show_progress_bar=True, batch_size=32)

print(f'Done! Embedding matrix shape: {embeddings.shape}')
# Expected: (394, 384)  →  394 items, 384-dim vectors

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded.
Generating embeddings for 394 menu items...


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Done! Embedding matrix shape: (394, 384)


## Step 6 — Save to `embeddings.pkl`
We save both the embeddings array and the metadata DataFrame together so retrieval.py can load everything it needs in one file.

In [9]:
import pickle

# Keep only the columns retrieval.py will actually need
metadata_cols = [
    'item_id', 'location_id',
    'name_item',           # original item name (for display)
    'name_location',       # original location name (for display)
    'meal_period',
    'description_item',    # original description (for display)
    'generated_tags',
    'cuisine_clean',
    'hours',
    'location',
    'text_blob'
]

# Some column names may vary — keep whichever exist
available_cols = [c for c in metadata_cols if c in merged_df.columns]
metadata_df = merged_df[available_cols].copy()

payload = {
    'embeddings': embeddings,   # numpy array (394, 384)
    'metadata':   metadata_df,  # DataFrame with item info
    'model_name': 'all-MiniLM-L6-v2'  # record which model was used
}

OUTPUT_PATH = 'embeddings.pkl'
with open(OUTPUT_PATH, 'wb') as f:
    pickle.dump(payload, f)

print(f'Saved to {OUTPUT_PATH}')
print(f'File size: {__import__("os").path.getsize(OUTPUT_PATH) / 1024:.1f} KB')

Saved to embeddings.pkl
File size: 765.3 KB


In [10]:
# Download embeddings.pkl to your local machine
# then put it in your project's data/ folder
from google.colab import files
files.download('embeddings.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query: str, top_k: int = 5) -> pd.DataFrame:
    """
    Given a natural language query, return the top_k most relevant menu items.
    This is the core retrieval function — same logic will go in src/retrieval.py.
    """
    # Embed the query using the same model
    query_vec = model.encode([query])  # shape: (1, 384)

    # Cosine similarity between query and all menu item embeddings
    scores = cosine_similarity(query_vec, embeddings)[0]  # shape: (394,)

    # Get indices of top_k highest scores
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = metadata_df.iloc[top_indices].copy()
    results['similarity_score'] = scores[top_indices]
    return results


# ---- Test queries ----
test_queries = [
    "I want something warm and comforting for breakfast",
    "I'm craving something spicy",
    "I want a healthy vegan lunch",
    "late night snack, something quick",
    "I want sushi",
]

for q in test_queries:
    print(f"\n Query: '{q}'")
    print('-' * 60)
    results = retrieve(q, top_k=3)
    for _, row in results.iterrows():
        # Use .get() to handle any column name differences
        item_name = row.get('name_item', row.get('name', 'Unknown item'))
        loc_name  = row.get('name_location', row.get('name', 'Unknown location'))
        print(f"  {item_name} @ {loc_name}  (score: {row['similarity_score']:.3f})")


 Query: 'I want something warm and comforting for breakfast'
------------------------------------------------------------
  Bacon, Egg, and Cheese Sandwich @ Beyu Blue Coffee  (score: 0.492)
  Egg and Cheese Sandwich @ Beyu Blue Coffee  (score: 0.487)
  Bacon, Egg, and Cheese Croissant @ Cafe  (score: 0.487)

 Query: 'I'm craving something spicy'
------------------------------------------------------------
  Spicy Il Forno @ Il Forno  (score: 0.463)
  Spicy Tuba Yubu @ Gyotaku  (score: 0.461)
  Spicy Cauliflower Wrap @ Sprout  (score: 0.455)

 Query: 'I want a healthy vegan lunch'
------------------------------------------------------------
  Sides @ Sprout  (score: 0.700)
  3 Composed Salad @ Sprout  (score: 0.668)
  Sides @ The Skillet  (score: 0.655)

 Query: 'late night snack, something quick'
------------------------------------------------------------
  Yogurt and Oatmeal Bar Topppings @ Sprout  (score: 0.443)
  Apple and Fennel Salad @ JB's Roasts & Chops  (score: 0.435)
  Mozz